# BART lemmatizer for Akkadian (Babylonian, 2nd millennium BCE)

A replicable workflow that fine-tunes BART to annotate each Akkadian word with its lemma and part of speech (POS), and adds an English gloss from a lemma lookup. It uses the same data and splits as Aleksi Sahala's BabyLemmatizer (asahala/BabyLemmatizer-Babylonian-2nd on Hugging Face), so the scores compare directly.

Pipeline: download data → check it → build examples → train → evaluate on dev → evaluate once on test → compare with BabyLemmatizer.

To rerun: Runtime → Change runtime type → GPU, then Runtime → Run all.

## 1. Setup and settings

Every choice lives in this cell. To try another dialect, change `dataset` and `hf_repo`. To repeat the run with another random seed, change `seed`; each seed gets its own output folder.

In [1]:
!pip -q install transformers datasets accelerate

import os, json, glob, platform
import numpy as np, pandas as pd, torch, transformers
from tqdm.auto import tqdm
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq,
                          Seq2SeqTrainingArguments, Seq2SeqTrainer)
from google.colab import drive

SETTINGS = {
    "dataset": "babylonian-2nd",
    "hf_repo": "asahala/BabyLemmatizer-Babylonian-2nd",
    "model_name": "facebook/bart-base",
    "max_source_len": 128,
    "max_target_len": 64,
    "learning_rate": 5e-5,
    "batch_size": 32,
    "epochs": 5,
    "seed": 2,
    "dev_eval_limit": 2000,     # dev words scored after each epoch, to pick the best epoch
    "num_beams": 5,
    "lookup_min_share": 0.9,    # trust training data for spellings with one analysis >= 90% of the time
    "drive_root": "/content/drive/MyDrive/bart_lemmatizer",
}

# asahala's published test scores
BENCHMARK = {"All lemma": 94.58, "All POS": 97.85, "All both": 93.87,
             "Unseen lemma": 71.33, "Unseen POS": 91.94, "Unseen both": 69.91}

drive.mount("/content/drive")
OUTPUT_DIR = f"{SETTINGS['drive_root']}/{SETTINGS['dataset']}/seed{SETTINGS['seed']}"
os.makedirs(OUTPUT_DIR, exist_ok=True)
transformers.set_seed(SETTINGS["seed"])

print("Python", platform.python_version(), "| torch", torch.__version__,
      "| transformers", transformers.__version__,
      "| GPU", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("Results go to", OUTPUT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Python 3.13.15 | torch 2.11.0+cu130 | transformers 5.20.1 | GPU Tesla T4
Results go to /content/drive/MyDrive/bart_lemmatizer/babylonian-2nd/seed2


## 2. Data

asahala's package contains the full annotations in CoNLL-U format (one word per line), already split into train, dev and test by text, so no tablet appears in two splits. Columns used: the written form, lemma, POS, text ID, English gloss, dialect, and a context window of two words on each side.

In [2]:
DATA_DIR = SETTINGS["dataset"]
if not os.path.exists(DATA_DIR):
    !wget -q https://huggingface.co/{SETTINGS['hf_repo']}/resolve/main/{SETTINGS['dataset']}.tar.gz
    !tar -xzf {SETTINGS['dataset']}.tar.gz

COLUMNS = {1: "form", 2: "lemma", 4: "pos", 9: "text_id", 10: "gloss", 12: "dialect", 13: "context"}

def read_conllu(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            if line and not line.startswith("#"):
                parts = line.split("\t")
                rows.append({name: parts[i] for i, name in COLUMNS.items()})
    return pd.DataFrame(rows)

splits = {name: read_conllu(f"{DATA_DIR}/conllu/{name}.conllu") for name in ["train", "dev", "test"]}
for name, df in splits.items():
    print(f"{name:5}: {len(df):,} words from {df['text_id'].nunique():,} texts")
splits["train"].head(5)

train: 92,323 words from 1,104 texts
dev  : 14,544 words from 138 texts
test : 12,498 words from 138 texts


,form,lemma,pos,text_id,gloss,dialect,context
0,a-na-ku,anāku,IP,aemw/alalakh/idrimi/P500443,I,middle babylonian,<SOU>|<SOU>|a-na-ku|{m}id-ri-mi|DUMU
1,{m}id-ri-mi,Idrimi,RN,aemw/alalakh/idrimi/P500443,Idrimi,middle babylonian,<SOU>|a-na-ku|{m}id-ri-mi|DUMU|{m}DINGIR-i-lim-ma
2,DUMU,māru,N,aemw/alalakh/idrimi/P500443,son,middle babylonian,a-na-ku|{m}id-ri-mi|DUMU|{m}DINGIR-i-lim-ma|ARAD
3,{m}DINGIR-i-lim-ma,Ili-ilimma,PN,aemw/alalakh/idrimi/P500443,Ili-ilimma,middle babylonian,{m}id-ri-mi|DUMU|{m}DINGIR-i-lim-ma|ARAD|{d}IM
4,ARAD,wardu,N,aemw/alalakh/idrimi/P500443,slave,middle babylonian,DUMU|{m}DINGIR-i-lim-ma|ARAD|{d}IM|{d}he₂-bat


### Data checks

Two facts about this data shape how it is evaluated:

- About half the tokens are not words: broken fragments (POS `u`) and numbers (`n`), with no lemma (`_`). They are easy to "get right", so accuracy is also reported on real words only.
- The gloss is missing almost only on those non-words. On real words it is nearly always present, and most lemma + POS pairs have a single gloss, so the gloss can be filled from a lookup instead of being predicted.

In [3]:
train = splits["train"]
real = ~train["pos"].isin(["u", "n"])
has_gloss = train["gloss"] != "_"
per_lemma = train[has_gloss].groupby(["lemma", "pos"])["gloss"].nunique()

print(f"Broken fragments or numbers: {(~real).mean():.1%} of training tokens")
print(f"Gloss missing on real words: {(~has_gloss[real]).mean():.1%}")
print(f"Lemma + POS pairs with exactly one gloss: {(per_lemma == 1).mean():.1%}")

Broken fragments or numbers: 51.9% of training tokens
Gloss missing on real words: 0.4%
Lemma + POS pairs with exactly one gloss: 97.5%


## 3. Training examples

Each word becomes one input → target pair.

- Input: the two words before, the word itself in brackets, and the two words after. The target word is spelled out character by character, so BART reads every sign instead of splitting the word into subword pieces learned from English.
- Target: the lemma spelled out, then `|`, then the POS.

```
EN-ia DINGIR-ia [ { d } U T U - i a ] um-ma {m}a-zi-ri  →  Š a m a š O R Š a p š u O R Š i m i g e | DN
```

Spelling the word out was the single largest improvement (+5.7 points on unseen words on dev; see Findings).

In [4]:
def spaced(text):
    return " ".join(text)

def make_source(context):
    w = context.split("|")
    return f"{w[0]} {w[1]} [ {spaced(w[2])} ] {w[3]} {w[4]}"

def make_target(row):
    return f"{spaced(row['lemma'])} | {row['pos']}"

def split_target(text):
    """'m ā r u | N' -> ('māru', 'N')"""
    if " | " in text:
        lemma, pos = text.rsplit(" | ", 1)
    else:
        lemma, pos = text, ""
    return lemma.replace(" ", ""), pos.strip()

for df in splits.values():
    df["source"] = df["context"].map(make_source)
    df["target"] = df.apply(make_target, axis=1)

splits["train"][["source", "target", "gloss"]].head(5)

,source,target,gloss
0,<SOU> <SOU> [ a - n a - k u ] {m}id-ri-mi DUMU,a n ā k u | IP,I
1,<SOU> a-na-ku [ { m } i d - r i - m i ] DUMU {...,I d r i m i | RN,Idrimi
2,a-na-ku {m}id-ri-mi [ D U M U ] {m}DINGIR-i-li...,m ā r u | N,son
3,{m}id-ri-mi DUMU [ { m } D I N G I R - i - l i...,I l i - i l i m m a | PN,Ili-ilimma
4,DUMU {m}DINGIR-i-lim-ma [ A R A D ] {d}IM {d}h...,w a r d u | N,slave


## 4. Training

Fine-tunes `facebook/bart-base`. Two details matter:

- Padding in the targets is hidden from the loss (`DataCollatorForSeq2Seq` pads labels with -100). Without this, a model can learn to output nothing.
- Training is checked on dev, never on train. After each epoch it scores a fixed sample of dev words and keeps the best epoch, saved to `OUTPUT_DIR/best`.

In [5]:
tokenizer = AutoTokenizer.from_pretrained(SETTINGS["model_name"])
model = AutoModelForSeq2SeqLM.from_pretrained(SETTINGS["model_name"])

def to_dataset(df, limit=None):
    df = df[["source", "target"]]
    if limit:
        df = df.sample(n=min(limit, len(df)), random_state=SETTINGS["seed"])
    return Dataset.from_pandas(df.reset_index(drop=True))

def tokenize(batch):
    encoded = tokenizer(batch["source"], max_length=SETTINGS["max_source_len"], truncation=True)
    encoded["labels"] = tokenizer(text_target=batch["target"],
                                  max_length=SETTINGS["max_target_len"], truncation=True)["input_ids"]
    return encoded

train_ds = to_dataset(splits["train"]).map(tokenize, batched=True, remove_columns=["source", "target"])
dev_ds = to_dataset(splits["dev"], SETTINGS["dev_eval_limit"]).map(tokenize, batched=True, remove_columns=["source", "target"])

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    if isinstance(preds, tuple):
        preds = preds[0]
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    pairs = [(split_target(p), split_target(g))
             for p, g in zip(tokenizer.batch_decode(preds, skip_special_tokens=True),
                             tokenizer.batch_decode(labels, skip_special_tokens=True))]
    return {"lemma_acc": float(np.mean([p[0] == g[0] for p, g in pairs])),
            "pos_acc": float(np.mean([p[1] == g[1] for p, g in pairs])),
            "both_acc": float(np.mean([p == g for p, g in pairs]))}

args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=SETTINGS["learning_rate"],
    per_device_train_batch_size=SETTINGS["batch_size"],
    per_device_eval_batch_size=SETTINGS["batch_size"],
    num_train_epochs=SETTINGS["epochs"],
    seed=SETTINGS["seed"],
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="both_acc",
    greater_is_better=True,
    predict_with_generate=True,
    generation_max_length=SETTINGS["max_target_len"],
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    report_to="none",
)
trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=dev_ds,
                         data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
                         compute_metrics=compute_metrics, processing_class=tokenizer)
trainer.train()
trainer.save_model(f"{OUTPUT_DIR}/best")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/best")
model.eval();

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

Map:   0%|          | 0/92323 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Lemma Acc,Pos Acc,Both Acc
1,0.093475,0.075802,0.937500,0.979000,0.932500
2,0.067905,0.060496,0.953500,0.981000,0.949500
3,0.037673,0.055209,0.957000,0.980000,0.954000
4,0.023080,0.053294,0.965500,0.990000,0.963500
5,0.019731,0.054987,0.968500,0.991000,0.966500


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [6]:
# Optional: skip training above and run only this to reload a saved model
tokenizer = AutoTokenizer.from_pretrained(f"{OUTPUT_DIR}/best")
model = AutoModelForSeq2SeqLM.from_pretrained(f"{OUTPUT_DIR}/best")
model = model.to("cuda" if torch.cuda.is_available() else "cpu").eval()

Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

## 5. Evaluation tools

- `predict` writes the model's answer for every word, with beam search.
- Seen-spelling lookup: a spelling that had the same lemma and POS at least 90% of the time in training is answered from training data instead of the model (BabyLemmatizer's "post-correct" step plays a similar role). Built from training data only.
- Gloss lookup: lemma + POS → its most common gloss in training.
- `score` reports lemma, POS and both for three groups: All tokens (as BabyLemmatizer reports it); Real words (excluding broken fragments and numbers); and Unseen spellings, using asahala's own list (`lex/<split>-types-oov.xlit`) so the definition matches his exactly.

In [7]:
def predict(df, num_beams=SETTINGS["num_beams"], batch_size=128):
    answers = []
    for i in tqdm(range(0, len(df), batch_size)):
        batch = df["source"].iloc[i:i + batch_size].tolist()
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True,
                           max_length=SETTINGS["max_source_len"]).to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_length=SETTINGS["max_target_len"], num_beams=num_beams)
        answers += [split_target(t) for t in tokenizer.batch_decode(out, skip_special_tokens=True)]
    return answers

counts = train.groupby(["form", "lemma", "pos"]).size().reset_index(name="n")
counts["share"] = counts["n"] / counts.groupby("form")["n"].transform("sum")
top = counts.sort_values("n", ascending=False).drop_duplicates("form").set_index("form")
seen_lookup = top[top["share"] >= SETTINGS["lookup_min_share"]]

gloss_lookup = (train[has_gloss].groupby(["lemma", "pos"])["gloss"]
                .agg(lambda g: g.value_counts().index[0]))

def apply_lookup(df, answers):
    lemmas, poses = [a[0] for a in answers], [a[1] for a in answers]
    for i, form in enumerate(df["form"]):
        if form in seen_lookup.index:
            lemmas[i], poses[i] = seen_lookup.at[form, "lemma"], seen_lookup.at[form, "pos"]
    return pd.Series(lemmas, index=df.index), pd.Series(poses, index=df.index)

def unseen_forms(split_name):
    path = f"{DATA_DIR}/lex/{split_name}-types-oov.xlit"
    return {line.split("\t")[0] for line in open(path, encoding="utf-8") if line.strip()}

def score(df, lemmas, poses, split_name):
    lemma_ok, pos_ok = lemmas == df["lemma"], poses == df["pos"]
    groups = {"All": df["pos"].notna(),
              "Real": ~df["pos"].isin(["u", "n"]),
              "Unseen": df["form"].isin(unseen_forms(split_name))}
    result = {}
    for g, mask in groups.items():
        for name, ok in [("lemma", lemma_ok), ("POS", pos_ok), ("both", lemma_ok & pos_ok)]:
            result[f"{g} {name}"] = round(100 * ok[mask].mean(), 2)
    glosses = pd.Series([gloss_lookup.get((l, p), "_") for l, p in zip(lemmas, poses)], index=df.index)
    gold = groups["Real"] & (df["gloss"] != "_")
    result["Real gloss"] = round(100 * (glosses == df["gloss"])[gold].mean(), 2)
    return result

## 6. Dev evaluation

Settings are chosen on dev. The test set is used once, in the next section.

In [8]:
dev = splits["dev"]
dev_answers = predict(dev)
dev_scores = score(dev, *apply_lookup(dev, dev_answers), "dev")
pd.Series(dev_scores, name="dev")

  0%|          | 0/114 [00:00<?, ?it/s]

,dev
All lemma,96.49
All POS,98.56
All both,96.12
Real lemma,91.82
Real POS,96.71
Real both,90.96
Unseen lemma,68.73
Unseen POS,88.68
Unseen both,67.48
Real gloss,87.52


## 7. Test evaluation (run once) and comparison with BabyLemmatizer

In [9]:
test = splits["test"]
test_answers = predict(test)
test_lemmas, test_poses = apply_lookup(test, test_answers)
test_scores = score(test, test_lemmas, test_poses, "test")

unseen = test["form"].isin(unseen_forms("test"))
print(f"Unseen rate over non-broken words: {100 * unseen[test['pos'] != 'u'].mean():.2f}%  (asahala: 13.04%)")

comparison = pd.DataFrame({"BART (this run)": test_scores, "BabyLemmatizer": pd.Series(BENCHMARK)})
comparison["Difference"] = (comparison["BART (this run)"] - comparison["BabyLemmatizer"]).round(2)
display(comparison)

test.assign(pred_lemma=test_lemmas, pred_pos=test_poses).to_csv(f"{OUTPUT_DIR}/test_predictions.csv", index=False)
json.dump({"settings": SETTINGS, "dev": dev_scores, "test": test_scores},
          open(f"{OUTPUT_DIR}/results.json", "w"), indent=2, ensure_ascii=False)
print("Saved predictions and results to", OUTPUT_DIR)

  0%|          | 0/98 [00:00<?, ?it/s]

Unseen rate over non-broken words: 13.05%  (asahala: 13.04%)


,BART (this run),BabyLemmatizer,Difference
All POS,98.88,97.85,1.03
All both,96.54,93.87,2.67
All lemma,96.83,94.58,2.25
Real POS,97.81,NaN,NaN
Real both,93.08,NaN,NaN
Real gloss,90.96,NaN,NaN
Real lemma,93.63,NaN,NaN
Unseen POS,92.22,91.94,0.28
Unseen both,69.22,69.91,-0.69
Unseen lemma,69.90,71.33,-1.43


Saved predictions and results to /content/drive/MyDrive/bart_lemmatizer/babylonian-2nd/seed2


### Across seeds

After running the notebook with several seeds, this collects every results.json for this dataset and reports the mean and range.

In [10]:
runs = [json.load(open(p)) for p in glob.glob(f"{SETTINGS['drive_root']}/{SETTINGS['dataset']}/seed*/results.json")]
table = pd.DataFrame({f"seed {r['settings']['seed']}": r["test"] for r in runs})
table.assign(mean=table.mean(axis=1).round(2), min=table.min(axis=1), max=table.max(axis=1))

,seed 42,seed 1,seed 2,mean,min,max
All lemma,96.87,96.79,96.83,96.83,96.79,96.87
All POS,98.87,98.85,98.88,98.87,98.85,98.88
All both,96.54,96.49,96.54,96.52,96.49,96.54
Real lemma,93.69,93.55,93.63,93.62,93.55,93.69
Real POS,97.78,97.75,97.81,97.78,97.75,97.81
Real both,93.06,92.96,93.08,93.03,92.96,93.08
Unseen lemma,70.35,69.22,69.90,69.82,69.22,70.35
Unseen POS,92.11,92.11,92.22,92.15,92.11,92.22
Unseen both,69.33,68.32,69.22,68.96,68.32,69.33
Real gloss,90.96,90.85,90.96,90.92,90.85,90.96


## 8. Results (test set, 3 seeds: 42, 1, 2)

| Metric | Seed 42 | Seed 1 | Seed 2 | Mean | BabyLemmatizer | Mean − BabyLemmatizer |
|---|---|---|---|---|---|---|
| All lemma | 96.87 | 96.79 | 96.83 | 96.83 | 94.58 | +2.25 |
| All POS | 98.87 | 98.85 | 98.88 | 98.87 | 97.85 | +1.02 |
| All both | 96.54 | 96.49 | 96.54 | 96.52 | 93.87 | +2.65 |
| Unseen lemma | 70.35 | 69.22 | 69.90 | 69.82 | 71.33 | −1.51 |
| Unseen POS | 92.11 | 92.11 | 92.22 | 92.15 | 91.94 | +0.21 |
| Unseen both | 69.33 | 68.32 | 69.22 | 68.96 | 69.91 | −0.95 |
| Real lemma | 93.69 | 93.55 | 93.63 | 93.62 | — | — |
| Real POS | 97.78 | 97.75 | 97.81 | 97.78 | — | — |
| Real both | 93.06 | 92.96 | 93.08 | 93.03 | — | — |
| Real gloss | 90.96 | 90.85 | 90.96 | 90.92 | — | — |

- Unseen rate is 13.05% (asahala: 13.04%), so the splits match.
- "Real" leaves out broken words and numbers. BabyLemmatizer doesn't report Real or gloss.
- "Both" means the lemma and the POS are both correct.

## 9. Findings

1. **BART beats BabyLemmatizer overall.** Across 3 seeds it scores about 2.7 points higher on lemma+POS (96.52 vs 93.87). The overall scores barely change between seeds (by 0.1 or less).
2. **On unseen words it is close but slightly behind.** Lemma+POS is about 1 point lower (68.96 vs 69.91), and lemma alone is about 1.5 points lower. POS on unseen words is about the same, slightly higher. Unseen scores vary by about 1 point between seeds, so one seed alone could be misleading.
3. **How the input is split up matters most for new words.** BART's normal tokenizer cuts text into chunks built for English, which breaks up unfamiliar Akkadian spellings. Spelling each word out letter by letter closed most of the gap on unseen words (+5.7 points on unseen words on dev). This points to a character-level model like ByT5 as the next thing to try.
4. **It adds English glosses.** About 91% of real words get the correct English meaning, which BabyLemmatizer does not provide.

## 10. Limitations

- **One dialect only.** These results are for Babylonian (2nd millennium). Other dialects (Babylonian 1st millennium, Middle Assyrian, Neo-Assyrian) have not been tested yet.
- **Glosses come from a lookup, not the model.** The gloss is matched from the training data using the predicted lemma and POS. If the lemma is wrong, the gloss is wrong, and lemmas never seen in training get no gloss.
- **The BabyLemmatizer numbers are its published results.** I did not rerun it myself. The splits and unseen-word list are the same, so the comparison is fair, but differences in how scores are counted could still exist.
- **Only one model size and few settings were tried.** I used bart-base with default-style settings, so larger models or more tuning might do better.
- **Three seeds.** That's enough to see that the overall result is stable, but the unseen-word gap (about 1 point) is about as big as the variation between seeds.